"""
MDI3003 Lab 04 - Customer Segment Prediction
Dataset: UCI Online Retail II (Dataset C, Research Extension per lab manual)

IMPORTANT (per manual section 7.7 / Dataset-use rule):
Online Retail II does NOT ship with predefined customer segment labels.
The manual requires a "two-stage extension with a frozen, documented label-construction
procedure" before any supervised model can be trained. That is implemented below:

STAGE 1 (label construction, frozen BEFORE modeling):
    Standard RFM (Recency, Frequency, Monetary) scoring is used to assign every customer
    to one of four business segments: Champions, Loyal Customers, At Risk, Hibernating/Lost.

STAGE 2 (supervised prediction):
    A SEPARATE, non-circular feature set (country, average unit price, bulk-buy ratio,
    average basket size, product diversity, return rate, tenure, weekend ratio) is used
    to predict the Stage 1 label. The raw R, F, M values themselves are NOT used as
    predictors, because doing so would let the model trivially reconstruct the label it
    was built from (label circularity, explicitly flagged as a common mistake in the manual).
"""

In [1]:
from pathlib import Path
import json, platform, sys, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sklearn
from joblib import dump

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, KBinsDiscretizer
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import GaussianNB, CategoricalNB, BernoulliNB
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score
)

In [2]:
SEED = 42
np.random.seed(SEED)

In [3]:
OUT = Path("C:/FallSemester/C2 - Advanced Predictive Analytics/lab-DA4")
for d in ["figures", "models", "artifacts", "results"]:
    (OUT / d).mkdir(parents=True, exist_ok=True)

In [4]:
versions = {
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
}
(OUT / "artifacts" / "versions.json").write_text(json.dumps(versions, indent=2))
print(versions)

{'python': '3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26200-SP0', 'pandas': '2.3.3', 'numpy': '2.3.5', 'scikit_learn': '1.7.2'}


In [5]:
DATA_PATH = Path("C:/FallSemester/C2 - Advanced Predictive Analytics/lab-DA4/online_retail_II.xlsx")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

In [6]:
df1 = pd.read_excel(DATA_PATH, sheet_name="Year 2009-2010")
df2 = pd.read_excel(DATA_PATH, sheet_name="Year 2010-2011")
raw = pd.concat([df1, df2], ignore_index=True)
print("Raw transaction shape:", raw.shape)

Raw transaction shape: (1067371, 8)


In [7]:
dataset_card = {
    "dataset_name": "UCI Online Retail II",
    "source": "https://archive.ics.uci.edu/dataset/502/online+retail+ii",
    "doi": "10.24432/C5CG6D",
    "licence": "UCI Machine Learning Repository, public research/educational use",
    "record_count_raw": int(raw.shape[0]),
    "feature_count_raw": int(raw.shape[1]),
    "date_range": [str(raw["InvoiceDate"].min()), str(raw["InvoiceDate"].max())],
    "missing_customer_id_percent": round(float(raw["Customer ID"].isna().mean() * 100), 2),
    "unique_countries": int(raw["Country"].nunique()),
    "sensitive_attributes": "Country (geographic origin) is the only quasi-demographic field; "
                             "no age, gender, income, or other personal demographic data is present.",
    "privacy_and_intended_use": "Transaction-level data with a numeric Customer ID (no name, "
                                 "address, or payment data). Used here strictly for an academic "
                                 "customer-segmentation exercise.",
    "direct_suitability_per_manual": "RESEARCH EXTENSION. No predefined segment label exists; "
                                      "a frozen label-construction procedure is required and is "
                                      "documented in label_definition.md.",
}
(OUT / "artifacts" / "dataset_card.json").write_text(json.dumps(dataset_card, indent=2))

962

In [8]:
raw["IsCancellation"] = raw["Invoice"].astype(str).str.startswith("C")
raw_valid_id = raw.dropna(subset=["Customer ID"]).copy()
raw_valid_id["Customer ID"] = raw_valid_id["Customer ID"].astype(int)

sales = raw_valid_id[(~raw_valid_id["IsCancellation"]) &
                      (raw_valid_id["Quantity"] > 0) &
                      (raw_valid_id["Price"] > 0)].copy()
sales["LineTotal"] = sales["Quantity"] * sales["Price"]
sales["IsWeekend"] = sales["InvoiceDate"].dt.dayofweek.isin([5, 6])

print("Sales line rows used:", sales.shape[0], "of", raw.shape[0])

Sales line rows used: 805549 of 1067371


In [9]:
snapshot_date = raw_valid_id["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = sales.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("LineTotal", "sum"),
).reset_index()

In [10]:
rfm = rfm[rfm["Monetary"] > 0].copy()

In [11]:
def score_quartile(series, ascending):
    ranks = series.rank(method="first", ascending=ascending)
    return pd.qcut(ranks, 4, labels=[1, 2, 3, 4]).astype(int)

In [12]:
rfm["R_score"] = score_quartile(rfm["Recency"], ascending=False)  
rfm["F_score"] = score_quartile(rfm["Frequency"], ascending=True) 
rfm["M_score"] = score_quartile(rfm["Monetary"], ascending=True)   
rfm["RFM_total"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

In [13]:
def label_from_score(s):
    if s >= 10:
        return "Champions"
    elif s >= 7:
        return "Loyal Customers"
    elif s >= 4:
        return "At Risk"
    else:
        return "Hibernating or Lost"

In [14]:
rfm["customer_segment"] = rfm["RFM_total"].apply(label_from_score)

In [15]:
label_definition = """# Frozen Label-Construction Procedure (Stage 1)

Dataset: UCI Online Retail II. This dataset has NO predefined customer-segment label, so one
was constructed using a standard, documented RFM (Recency, Frequency, Monetary) scoring rule,
BEFORE any supervised model was trained. This rule was frozen and not modified after seeing
model results.

Snapshot date: max(InvoiceDate) + 1 day = {snapshot}

For each customer (sales lines only; cancellations excluded):
  Recency  = days since the customer's most recent purchase
  Frequency = number of distinct invoices
  Monetary  = total amount spent (Quantity x Price, summed)

Each of R, F, M is scored 1 (worst) to 4 (best) using quartiles across all customers.
RFM_total = R_score + F_score + M_score  (range 3 to 12)

Segment mapping (frozen business rule):
  RFM_total 10-12  -> Champions
  RFM_total 7-9    -> Loyal Customers
  RFM_total 4-6    -> At Risk
  RFM_total 3      -> Hibernating or Lost

## Label-Provenance and Circularity Audit (manual section 7.5)
Recency, Frequency, and Monetary (and their R/F/M scores) directly define this label by
construction. They are therefore EXCLUDED from the Stage 2 predictor set, because including
them would let the classifier trivially reconstruct the label it was derived from
(label circularity). Stage 2 instead uses an independent behavioral/demographic/psychographic
proxy feature set (country, average unit price, bulk-buy ratio, average basket size, product
diversity, return rate, tenure, weekend purchase ratio) that is correlated with, but not
deterministic of, the RFM label.

## Psychographic Measurement Provenance (manual section 7.6)
This dataset contains no self-reported psychographic survey data. The "average unit price"
and "bulk-buy ratio" features used in Stage 2 are BEHAVIORALLY INFERRED proxies for price
sensitivity and deal-seeking attitude, not directly measured psychographic constructs. They
should be interpreted with caution and are documented here as inferred, not ground truth.
""".format(snapshot=snapshot_date)

In [16]:
(OUT / "artifacts" / "label_definition.md").write_text(label_definition)
print(rfm["customer_segment"].value_counts())

customer_segment
Loyal Customers        1787
At Risk                1786
Champions              1728
Hibernating or Lost     577
Name: count, dtype: int64


In [17]:
country_mode = sales.groupby("Customer ID")["Country"].agg(lambda x: x.mode().iloc[0])

In [18]:
basket = sales.groupby("Customer ID").agg(
    avg_unit_price=("Price", "mean"),
    avg_basket_size=("Quantity", "mean"),
    product_diversity=("StockCode", pd.Series.nunique),
    weekend_ratio=("IsWeekend", "mean"),
    first_purchase=("InvoiceDate", "min"),
    last_purchase=("InvoiceDate", "max"),
).reset_index()
basket["tenure_days"] = (basket["last_purchase"] - basket["first_purchase"]).dt.days

In [19]:
bulk = sales.copy()
bulk["is_bulk"] = bulk["Quantity"] >= 12
bulk_ratio = bulk.groupby("Customer ID")["is_bulk"].mean().rename("bulk_buy_ratio").reset_index()

In [20]:
all_invoices = raw_valid_id.groupby("Customer ID")["Invoice"].nunique().rename("all_invoice_count")
cancel_invoices = raw_valid_id[raw_valid_id["IsCancellation"]].groupby("Customer ID")["Invoice"].nunique().rename("cancel_invoice_count")
return_df = pd.concat([all_invoices, cancel_invoices], axis=1).fillna(0)
return_df["return_rate"] = return_df["cancel_invoice_count"] / return_df["all_invoice_count"].replace(0, np.nan)

In [21]:
return_df = return_df.reset_index()[["Customer ID", "return_rate"]]

In [22]:
customers = rfm[["Customer ID", "customer_segment", "Recency", "Frequency", "Monetary", "RFM_total"]].merge(
    country_mode.rename("country"), on="Customer ID", how="left"
).merge(
    basket[["Customer ID", "avg_unit_price", "avg_basket_size", "product_diversity",
            "weekend_ratio", "tenure_days"]], on="Customer ID", how="left"
).merge(
    bulk_ratio, on="Customer ID", how="left"
).merge(
    return_df, on="Customer ID", how="left"
)
customers["return_rate"] = customers["return_rate"].fillna(0.0)

In [23]:
top_countries = customers["country"].value_counts().nlargest(8).index
customers["country"] = customers["country"].where(customers["country"].isin(top_countries), "Other")

In [24]:
customers.to_csv(OUT / "results" / "customer_level_dataset.csv", index=False)
print("Customer-level dataset shape:", customers.shape)

Customer-level dataset shape: (5878, 14)


In [25]:
ID_COL = "Customer ID"
TARGET = "customer_segment"
DEMOGRAPHIC = ["country"]
PSYCHOGRAPHIC = ["avg_unit_price", "bulk_buy_ratio"]
BEHAVIORAL = ["avg_basket_size", "product_diversity", "weekend_ratio", "tenure_days", "return_rate"]
ALL_FEATURES = DEMOGRAPHIC + PSYCHOGRAPHIC + BEHAVIORAL

In [26]:
feature_manifest = {
    "demographic": DEMOGRAPHIC,
    "psychographic": PSYCHOGRAPHIC,
    "behavioral": BEHAVIORAL,
    "all_features": ALL_FEATURES,
    "excluded_circular_features": ["Recency", "Frequency", "Monetary", "RFM_total",
                                    "R_score", "F_score", "M_score"],
    "exclusion_reason": "These fields deterministically define customer_segment and were "
                         "excluded to prevent label circularity (see label_definition.md).",
}
(OUT / "artifacts" / "feature_manifest.json").write_text(json.dumps(feature_manifest, indent=2))
print(json.dumps(feature_manifest, indent=2))

{
  "demographic": [
    "country"
  ],
  "psychographic": [
    "avg_unit_price",
    "bulk_buy_ratio"
  ],
  "behavioral": [
    "avg_basket_size",
    "product_diversity",
    "weekend_ratio",
    "tenure_days",
    "return_rate"
  ],
  "all_features": [
    "country",
    "avg_unit_price",
    "bulk_buy_ratio",
    "avg_basket_size",
    "product_diversity",
    "weekend_ratio",
    "tenure_days",
    "return_rate"
  ],
  "excluded_circular_features": [
    "Recency",
    "Frequency",
    "Monetary",
    "RFM_total",
    "R_score",
    "F_score",
    "M_score"
  ],
  "exclusion_reason": "These fields deterministically define customer_segment and were excluded to prevent label circularity (see label_definition.md)."
}


In [27]:
df = customers.copy()
assert df[TARGET].nunique() >= 2
assert not df[ID_COL].duplicated().any()

In [28]:
summary = pd.DataFrame({
    "dtype": df[ALL_FEATURES + [TARGET]].dtypes.astype(str),
    "missing_count": df[ALL_FEATURES + [TARGET]].isna().sum(),
    "missing_percent": 100 * df[ALL_FEATURES + [TARGET]].isna().mean(),
    "unique_count": df[ALL_FEATURES + [TARGET]].nunique(dropna=False),
})

In [29]:
summary.to_csv(OUT / "results" / "data_audit.csv")
print(summary)
print("Exact duplicate rows:", df.duplicated().sum())

                     dtype  missing_count  missing_percent  unique_count
country             object              0              0.0             9
avg_unit_price     float64              0              0.0          5689
bulk_buy_ratio     float64              0              0.0          2652
avg_basket_size    float64              0              0.0          4827
product_diversity    int64              0              0.0           463
weekend_ratio      float64              0              0.0          1467
tenure_days          int64              0              0.0           732
return_rate        float64              0              0.0           253
customer_segment    object              0              0.0             4
Exact duplicate rows: 0


In [30]:
class_counts = df[TARGET].value_counts().sort_index()
class_counts.to_csv(OUT / "results" / "class_distribution.csv")
ax = class_counts.plot(kind="bar", title="Customer Segment Distribution (RFM-derived)", color="#2c5f8a")
ax.set_xlabel("Segment"); ax.set_ylabel("Number of customers")
plt.tight_layout(); plt.savefig(OUT / "figures" / "class_distribution.png", dpi=180); plt.close()

In [31]:
usable = df.dropna(subset=[TARGET]).copy()
train_df, test_df = train_test_split(
    usable, test_size=0.20, random_state=SEED, stratify=usable[TARGET]
)

In [32]:
assert set(train_df[ID_COL]).isdisjoint(set(test_df[ID_COL]))
split_manifest = pd.concat([
    train_df[[ID_COL]].assign(split="train"),
    test_df[[ID_COL]].assign(split="test")
], ignore_index=True)
split_manifest.to_csv(OUT / "artifacts" / "split_manifest.csv", index=False)

In [33]:
X_train = train_df[ALL_FEATURES]; y_train = train_df[TARGET]
X_test = test_df[ALL_FEATURES]; y_test = test_df[TARGET]
print(X_train.shape, X_test.shape)

(4702, 8) (1176, 8)


In [34]:
numeric_cols = ["avg_unit_price", "bulk_buy_ratio", "avg_basket_size",
                 "product_diversity", "weekend_ratio", "tenure_days", "return_rate"]
nominal_cols = ["country"]
binary_cols = []
print("Numeric:", numeric_cols); print("Nominal:", nominal_cols)

Numeric: ['avg_unit_price', 'bulk_buy_ratio', 'avg_basket_size', 'product_diversity', 'weekend_ratio', 'tenure_days', 'return_rate']
Nominal: ['country']


In [35]:
class SafeOrdinalToNonNegative(BaseEstimator, TransformerMixin):
    """Encode known categories as 1..K and reserve 0 for unseen categories."""
    def fit(self, X, y=None):
        self.enc_ = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        self.enc_.fit(X)
        return self
    def transform(self, X):
        return self.enc_.transform(X).astype(int) + 1

numeric_binary = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("bins", KBinsDiscretizer(n_bins=4, encode="onehot", strategy="quantile")),
])
category_ohe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
bernoulli_preprocessor = ColumnTransformer([
    ("num_bins", numeric_binary, numeric_cols),
    ("cat", category_ohe, nominal_cols + binary_cols),
], remainder="drop")

cat_num = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("bins", KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile")),
])
cat_cat = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("safe_ordinal", SafeOrdinalToNonNegative()),
])
categorical_nb_preprocessor = ColumnTransformer([
    ("num", cat_num, numeric_cols),
    ("cat", cat_cat, nominal_cols + binary_cols),
], remainder="drop")

core_pipelines = {
    "Dummy_most_frequent": Pipeline([("prep", bernoulli_preprocessor),
                                      ("model", DummyClassifier(strategy="most_frequent"))]),
    "BernoulliNB": Pipeline([("prep", bernoulli_preprocessor),
                              ("model", BernoulliNB(alpha=1.0, binarize=0.0))]),
    "CategoricalNB_mixed": Pipeline([("prep", categorical_nb_preprocessor),
                                      ("model", CategoricalNB(alpha=1.0))]),
}

gaussian_preprocessor = Pipeline([("imputer", SimpleImputer(strategy="median"))])
core_pipelines["GaussianNB_numeric_only"] = Pipeline([
    ("prep", ColumnTransformer([("num", gaussian_preprocessor, numeric_cols)], remainder="drop")),
    ("model", GaussianNB(var_smoothing=1e-9)),
])

In [36]:
Xt_cat = categorical_nb_preprocessor.fit_transform(X_train)
assert np.nanmin(np.asarray(Xt_cat)) >= 0, "CategoricalNB received a negative category code."

C:\Users\Sarah Blessy\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
C:\Users\Sarah Blessy\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:397: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 4 are removed. Consider decreasing the number of bins.
  warnings.warn(
C:\Users\Sarah Blessy\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:397: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 5 are removed. Consider decreasing the number of bins.
  warnings.warn(
C:\Users\Sarah Blessy\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:397: User

In [37]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {"accuracy": "accuracy", "macro_f1": "f1_macro", "weighted_f1": "f1_weighted"}

In [38]:
rows = []
for name, pipe in core_pipelines.items():
    start = time.perf_counter()
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring,
                             return_train_score=False, n_jobs=1)
    elapsed = time.perf_counter() - start
    rows.append({
        "model": name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_sd": scores["test_accuracy"].std(ddof=1),
        "macro_f1_mean": scores["test_macro_f1"].mean(),
        "macro_f1_sd": scores["test_macro_f1"].std(ddof=1),
        "weighted_f1_mean": scores["test_weighted_f1"].mean(),
        "cv_time_seconds": elapsed,
    })
cv_results = pd.DataFrame(rows).sort_values("macro_f1_mean", ascending=False)
cv_results.to_csv(OUT / "results" / "cv_results.csv", index=False)
print(cv_results)

C:\Users\Sarah Blessy\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
C:\Users\Sarah Blessy\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:397: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 4 are removed. Consider decreasing the number of bins.
  warnings.warn(
C:\Users\Sarah Blessy\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:397: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 5 are removed. Consider decreasing the number of bins.
  warnings.warn(
C:\Users\Sarah Blessy\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:397: User

                     model  accuracy_mean  accuracy_sd  macro_f1_mean  \
1              BernoulliNB       0.612078     0.013203       0.578346   
2      CategoricalNB_mixed       0.631644     0.016969       0.551718   
3  GaussianNB_numeric_only       0.495533     0.009992       0.444715   
0      Dummy_most_frequent       0.303700     0.000483       0.116476   

   macro_f1_sd  weighted_f1_mean  cv_time_seconds  
1     0.009869          0.610539         0.498958  
2     0.015662          0.615710         0.412890  
3     0.017695          0.427273         0.244180  
0     0.000142          0.141496         0.462006  


In [39]:
ax = cv_results.set_index("model")[["accuracy_mean", "macro_f1_mean", "weighted_f1_mean"]].plot(
    kind="bar", figsize=(8, 5), title="Cross Validation Comparison (mean of 5 folds)")
plt.ylabel("Score"); plt.xticks(rotation=20, ha="right")
plt.tight_layout(); plt.savefig(OUT / "figures" / "cv_comparison.png", dpi=180); plt.close()

In [40]:
selected_name = cv_results.iloc[0]["model"]
selected_model = core_pipelines[selected_name]
print("Selected by mean CV macro F1:", selected_name)

Selected by mean CV macro F1: BernoulliNB


In [41]:
start = time.perf_counter()
selected_model.fit(X_train, y_train)
train_seconds = time.perf_counter() - start

C:\Users\Sarah Blessy\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
C:\Users\Sarah Blessy\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:397: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 4 are removed. Consider decreasing the number of bins.
  warnings.warn(
C:\Users\Sarah Blessy\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:397: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 5 are removed. Consider decreasing the number of bins.
  warnings.warn(
C:\Users\Sarah Blessy\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:397: User

In [ ]:
start = time.perf_counter()
y_pred = selected_model.predict(X_test)
inference_seconds = time.perf_counter() - start

In [43]:
report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
pd.DataFrame(report).T.to_csv(OUT / "results" / "classification_report.csv")
print(classification_report(y_test, y_pred, zero_division=0))

                     precision    recall  f1-score   support

            At Risk       0.58      0.56      0.57       357
          Champions       0.74      0.85      0.79       346
Hibernating or Lost       0.35      0.40      0.37       115
    Loyal Customers       0.59      0.49      0.54       358

           accuracy                           0.61      1176
          macro avg       0.56      0.58      0.57      1176
       weighted avg       0.61      0.61      0.61      1176



In [44]:
test_summary = pd.DataFrame([{
    "model": selected_name,
    "accuracy": accuracy_score(y_test, y_pred),
    "macro_f1": f1_score(y_test, y_pred, average="macro"),
    "weighted_f1": f1_score(y_test, y_pred, average="weighted"),
    "train_seconds": train_seconds,
    "inference_seconds": inference_seconds,
    "test_n": len(y_test),
}])

In [45]:
test_summary.to_csv(OUT / "results" / "test_summary.csv", index=False)
print(test_summary)

         model  accuracy  macro_f1  weighted_f1  train_seconds  \
0  BernoulliNB  0.609694  0.567736     0.605796        0.04953   

   inference_seconds  test_n  
0            0.01429    1176  


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, xticks_rotation=30, cmap="Blues")
plt.title(f"Confusion Matrix - {selected_name}")
plt.tight_layout(); plt.savefig(OUT / "figures" / "confusion_matrix.png", dpi=180); plt.close()

In [ ]:
cm_norm = confusion_matrix(y_test, y_pred, normalize="true")
labels_sorted = sorted(y_test.unique())
pd.DataFrame(cm_norm, index=labels_sorted, columns=labels_sorted).to_csv(
    OUT / "results" / "confusion_matrix_normalized.csv")

In [46]:
pred_df = test_df[[ID_COL, TARGET]].copy()
pred_df["predicted_segment"] = y_pred

In [47]:
if hasattr(selected_model, "predict_proba"):
    probs = selected_model.predict_proba(X_test)
    classes = selected_model.classes_
    pred_df["max_posterior"] = probs.max(axis=1)
    pred_df["confidence_category"] = pd.cut(
        pred_df["max_posterior"], bins=[-np.inf, .50, .75, np.inf],
        labels=["low_review", "moderate_review", "high"]
    )
    for i, cls in enumerate(classes):
        pred_df[f"prob_{cls}"] = probs[:, i]
else:
    pred_df["max_posterior"] = np.nan
    pred_df["confidence_category"] = "decision_score_only"

In [48]:
pred_df.to_csv(OUT / "results" / "test_predictions.csv", index=False)

In [49]:
errors = pred_df[pred_df[TARGET] != pred_df["predicted_segment"]].copy()
errors = errors.merge(test_df[[ID_COL] + ALL_FEATURES], on=ID_COL, how="left")
errors.to_csv(OUT / "results" / "error_analysis.csv", index=False)
print("Test errors:", len(errors), "of", len(pred_df))

dump(selected_model, OUT / "models" / "selected_pipeline.joblib")

Test errors: 459 of 1176


['C:\\FallSemester\\C2 - Advanced Predictive Analytics\\lab-DA4\\models\\selected_pipeline.joblib']

In [42]:


# ---------------------------------------------------------------------------
# A13. New-customer prediction helper
# ---------------------------------------------------------------------------
REQUIRED_INPUTS = ALL_FEATURES
def predict_customer_segment(customer_profile: dict) -> dict:
    missing = [c for c in REQUIRED_INPUTS if c not in customer_profile]
    if missing:
        raise ValueError(f"Missing mandatory fields: {missing}")
    one = pd.DataFrame([customer_profile], columns=REQUIRED_INPUTS)
    pred = selected_model.predict(one)[0]
    result = {"predicted_segment": str(pred)}
    if hasattr(selected_model, "predict_proba"):
        p = selected_model.predict_proba(one)[0]
        distribution = {str(c): float(v) for c, v in zip(selected_model.classes_, p)}
        confidence = float(p.max())
        review = "normal_review" if confidence >= .75 else (
            "explicit_review" if confidence >= .50 else "manual_analysis")
        result.update({"posterior_distribution": distribution,
                        "max_posterior": confidence,
                        "review_recommendation": review})
    return result

example_profile = X_test.iloc[0].to_dict()
example_result = predict_customer_segment(example_profile)
(OUT / "artifacts" / "example_new_customer_prediction.json").write_text(json.dumps(example_result, indent=2))
print(example_result)

# ---------------------------------------------------------------------------
# A14. Core acceptance tests
# ---------------------------------------------------------------------------
assert TARGET in df.columns
assert df[TARGET].nunique() >= 2
assert ID_COL not in ALL_FEATURES
assert set(train_df[ID_COL]).isdisjoint(set(test_df[ID_COL]))
assert set(np.unique(y_pred)).issubset(set(y_train.unique()))
assert (OUT / "models" / "selected_pipeline.joblib").exists()
assert (OUT / "results" / "cv_results.csv").exists()
assert (OUT / "results" / "test_predictions.csv").exists()
required_core = {"Dummy_most_frequent", "BernoulliNB", "CategoricalNB_mixed", "GaussianNB_numeric_only"}
assert required_core.issubset(set(cv_results["model"]))
reloaded = __import__("joblib").load(OUT / "models" / "selected_pipeline.joblib")
assert np.array_equal(
    reloaded.predict(X_test.head(5)), selected_model.predict(X_test.head(5))
)
print("Core acceptance tests: PASSED")



(OUT / "artifacts" / "README.md").write_text(readme)
print("\nDone. See lab04_outputs/ for all results.")


                     precision    recall  f1-score   support

            At Risk       0.58      0.56      0.57       357
          Champions       0.74      0.85      0.79       346
Hibernating or Lost       0.35      0.40      0.37       115
    Loyal Customers       0.59      0.49      0.54       358

           accuracy                           0.61      1176
          macro avg       0.56      0.58      0.57      1176
       weighted avg       0.61      0.61      0.61      1176

         model  accuracy  macro_f1  weighted_f1  train_seconds  \
0  BernoulliNB  0.609694  0.567736     0.605796        0.04953   

   inference_seconds  test_n  
0            0.01429    1176  
Test errors: 459 of 1176
{'predicted_segment': 'Loyal Customers', 'posterior_distribution': {'At Risk': 0.04453395739295047, 'Champions': 0.05187990704211831, 'Hibernating or Lost': 2.5168967034916366e-06, 'Loyal Customers': 0.9035836186682284}, 'max_posterior': 0.9035836186682284, 'review_recommendation': 'norma

NameError: name 'readme' is not defined